In [1]:
print('ML441 Assignment 3 staged experiment log')

ML441 Assignment 3 staged experiment log


In [2]:
import json
import os
import re
import subprocess
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / "configs/primary_protocol.json").exists():
    ROOT = ROOT.parent
if not (ROOT / "configs/primary_protocol.json").exists():
    raise RuntimeError("Start Jupyter in the assignment3 project or notebooks directory")
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
EXPORT = ROOT / "notebooks/exports"
EXPORT.mkdir(parents=True, exist_ok=True)
AUTO_COMMIT = os.environ.get("ML441_NOTEBOOK_COMMIT", "1") == "1"
PUSH_TO_REMOTE = os.environ.get("ML441_NOTEBOOK_PUSH", "0") == "1"
protocol = json.loads((ROOT / "configs/primary_protocol.json").read_text())

def run_module(name, *arguments):
    command = [sys.executable, "-m", name, *arguments]
    result = subprocess.run(command, cwd=ROOT, text=True, capture_output=True)
    print("$", " ".join(command))
    print(result.stdout[-1500:])
    if result.returncode:
        print(result.stderr[-3000:])
        raise RuntimeError(f"{name} failed with exit code {result.returncode}")
    return result

def publish_stage(label, paths):
    if not AUTO_COMMIT:
        print("Review stage before committing:", label, paths)
        return
    staged = subprocess.run(["git", "diff", "--cached", "--quiet"], cwd=ROOT)
    if staged.returncode != 0:
        raise RuntimeError("Commit or unstage existing staged changes before publishing this stage")
    subprocess.run(["git", "add", "--", *paths], cwd=ROOT, check=True)
    changes = subprocess.run(["git", "diff", "--cached", "--quiet"], cwd=ROOT)
    if changes.returncode:
        subprocess.run(["git", "commit", "-m", label], cwd=ROOT, check=True)
    else:
        print("No new changes for", label)
    if PUSH_TO_REMOTE:
        remote = subprocess.run(["git", "remote", "get-url", "origin"], cwd=ROOT,
                                text=True, capture_output=True)
        if remote.returncode:
            raise RuntimeError("Configure a Git remote named origin before enabling pushes")
        subprocess.run(["git", "push", "origin", "main"], cwd=ROOT, check=True)

print("Repository:", ROOT)
print("Local commits enabled:", AUTO_COMMIT, "| remote pushes enabled:", PUSH_TO_REMOTE)

Repository: /Users/devinmoolman/Desktop/semester 2/machine learning/assignment3
Local commits enabled: False | remote pushes enabled: False


In [3]:
print('Stage 1: Source snapshots and unit tests')

Stage 1: Source snapshots and unit tests


In [4]:
from src.download_data import SOURCES, sha256
from src.data import load_series

manifest = []
for key, filename in zip(protocol["datasets"], protocol["raw_sha256"]):
    path = ROOT / "data/raw" / filename
    if not path.exists():
        raise FileNotFoundError(f"Missing frozen source: {path}")
    actual_hash = sha256(path)
    expected_hash = protocol["raw_sha256"][filename]
    if actual_hash != expected_hash:
        raise ValueError(f"Source snapshot changed: {filename}")
    series = load_series(key)
    manifest.append({"dataset": key, "local_file": str(path.relative_to(ROOT)),
                     "source_url": SOURCES[filename], "sha256": actual_hash,
                     "target": series.source_note, "unit": series.unit,
                     "frequency": "day" if key == "power" else "month",
                     "modelled_length": len(series.values),
                     "modelled_missing": int(series.levels.isna().sum())})
manifest_table = pd.DataFrame(manifest)
manifest_table.to_csv(EXPORT / "dataset_manifest.csv", index=False)
display(manifest_table)
test_result = run_module("pytest", "-q", "tests")
passed = re.search(r"(\d+) passed", test_result.stdout)
if passed is None:
    raise AssertionError("Pytest did not report a passing test count")
(EXPORT / "unit_tests.json").write_text(json.dumps({
    "command": "python -m pytest -q tests", "exit_code": test_result.returncode,
    "passed_tests": int(passed.group(1))}, indent=2))

,dataset,local_file,source_url,sha256,target,unit,frequency,modelled_length,modelled_missing
0,co2,data/raw/co2_mm_mlo.csv,https://gml.noaa.gov/webdata/ccgg/trends/co2/c...,dcf0198658c87ebb5e3b2ce807fe3a52e1d5f000aa0479...,NOAA average column; source includes historica...,ppm,month,813,0
1,sunspots,data/raw/SN_m_tot_V2.0.csv,https://www.sidc.be/SILSO/DATA/SN_m_tot_V2.0.csv,6383f7c67fc2e67ef72a2035a1f2233d4564316560c49b...,SILSO version 2 monthly total; 2025 definitive...,sunspot number,month,1512,0
2,power,data/raw/household_power_consumption.zip,https://archive.ics.uci.edu/static/public/235/...,9f84b46ade8a2d8e1286ec4b2b6c2987a45a755c59f263...,UCI minute active power aggregated to daily me...,kW,day,1440,23
3,retail,data/raw/ons_j448.csv,https://www.ons.gov.uk/generator?format=csv&ur...,72fb945d0c16f4623f161d81df1d16353aaaec024f02f4...,ONS J448 unadjusted all-retailers-ex-fuel volu...,volume index,month,444,0
4,rain,data/raw/HadEWP_monthly_totals.txt,https://www.metoffice.gov.uk/hadobs/hadukp/dat...,202d0ba7360429539a0d3dd458b1c9b712d9b4dd8c3861...,Met Office HadEWP monthly England and Wales pr...,mm,month,1140,0


$ /opt/anaconda3/bin/python -m pytest -q tests
..........                                                               [100%]
10 passed in 2.12s



84

In [5]:
publish_stage("Archive frozen sources and unit-test evidence", [
    "data/raw", "notebooks/exports/dataset_manifest.csv",
    "notebooks/exports/unit_tests.json"])

Review stage before committing: Archive frozen sources and unit-test evidence ['data/raw', 'notebooks/exports/dataset_manifest.csv', 'notebooks/exports/unit_tests.json']


In [6]:
print('Stage 2: Development pilot')

Stage 2: Development pilot


In [7]:
if not (ROOT / "output/pilot/summary.json").exists():
    run_module("src.pilot")
pilot = json.loads((ROOT / "output/pilot/summary.json").read_text())
if not pilot["deterministic_repeat"]:
    raise AssertionError("Pilot repeat was not deterministic")
(EXPORT / "pilot_summary.json").write_text(json.dumps(pilot, indent=2))
display(pd.DataFrame(pilot["runs"]))

,architecture,seed,mae,best_epoch,runtime_seconds
0,elman,11,0.349264,400,2.095486
1,elman,23,0.349756,400,2.315512
2,jordan,11,0.617423,400,2.776088
3,jordan,23,0.619613,400,2.583731
4,multi,11,0.303326,370,2.927821
5,multi,23,0.297300,400,2.997263


In [8]:
publish_stage("Archive development-only pilot", [
    "output/pilot", "notebooks/exports/pilot_summary.json"])

Review stage before committing: Archive development-only pilot ['output/pilot', 'notebooks/exports/pilot_summary.json']


In [9]:
print('Stage 3: Development data audit')

Stage 3: Development data audit


In [10]:
if not (ROOT / "output/diagnostics/data_audit.json").exists():
    run_module("src.audit_data")
audit = json.loads((ROOT / "output/diagnostics/data_audit.json").read_text())
audit_table = pd.DataFrame([{
    "dataset": row["dataset"], "modelled_n": row["modelled_n"],
    "test_start": row["test_start"],
    "development_missing": row["development_missing"],
    "level_adf_p": row["development_level"]["adf_unit_root_null_p"],
    "level_kpss_p": row["development_level"]["kpss_level_stationarity_null_p"]}
    for row in audit])
audit_table.to_csv(EXPORT / "development_audit.csv", index=False)
display(audit_table)

,dataset,modelled_n,test_start,development_missing,level_adf_p,level_kpss_p
0,co2,813,2012-06-01,0,1.000000e+00,0.010000
1,sunspots,1512,2000-10-01,0,7.425327e-10,0.056057
2,power,1440,2010-02-11,11,2.753372e-02,0.100000
3,retail,444,2018-08-01,0,9.537990e-01,0.010000
4,rain,1140,2007-01-01,0,1.614439e-09,0.100000


In [11]:
publish_stage("Archive development-only data audit", [
    "output/diagnostics/data_audit.json", "notebooks/exports/development_audit.csv"])

Review stage before committing: Archive development-only data audit ['output/diagnostics/data_audit.json', 'notebooks/exports/development_audit.csv']


In [12]:
print('Stage 4: Primary development selection')

Stage 4: Primary development selection


In [13]:
run_module("src.run_experiment", "--phase", "selection")
freeze = json.loads((ROOT / "output/primary/all_selected_before_test.json").read_text())
validation_rows = []
selected_rows = []
for dataset in protocol["datasets"]:
    folder = ROOT / "output/primary/selection" / dataset
    for path in sorted(folder.glob("*_f*_s*.json")):
        row = json.loads(path.read_text())
        validation_rows.append({
            "dataset": dataset, "architecture": row["architecture"],
            "hidden": row["hidden"], "learning_rate": row["learning_rate"],
            "fold": row["fold"], "seed": row["seed"],
            "mae": row["dataset_metrics"]["mae"],
            "rmse": row["dataset_metrics"]["rmse"],
            "mase": row["dataset_metrics"]["mase"],
            "best_epoch": row["best_epoch"],
            "parameters": row["parameter_count"],
            "scaler_fitted_end": row["scaler"]["fitted_end"]})
    for architecture, choice in freeze[dataset]["selected"].items():
        selected_rows.append({"dataset": dataset, "architecture": architecture,
                              **choice})
if len(validation_rows) != 540 or len(selected_rows) != 15:
    raise AssertionError("Incomplete development selection")
pd.DataFrame(validation_rows).to_csv(EXPORT / "all_validation_runs.csv", index=False)
selected_table = pd.DataFrame(selected_rows)
selected_table.to_csv(EXPORT / "selected_configurations.csv", index=False)
display(selected_table)
print("Saved", len(validation_rows), "validation records and", len(selected_rows), "selections")

$ /opt/anaconda3/bin/python -m src.run_experiment --phase selection
18.993320478561433, 'parameter_count': 397, 'epochs': 11, 'cap_limited_fits': 0}}
SELECTED power {'elman': {'hidden': 18, 'learning_rate': 0.01, 'mean_validation_mae': 0.21155645523086763, 'parameter_count': 379, 'epochs': 17, 'cap_limited_fits': 0}, 'jordan': {'hidden': 96, 'learning_rate': 0.01, 'mean_validation_mae': 0.219081807296575, 'parameter_count': 385, 'epochs': 28, 'cap_limited_fits': 0}, 'multi': {'hidden': 18, 'learning_rate': 0.01, 'mean_validation_mae': 0.2122285399567472, 'parameter_count': 397, 'epochs': 15, 'cap_limited_fits': 0}}
SELECTED retail {'elman': {'hidden': 8, 'learning_rate': 0.01, 'mean_validation_mae': 1.3951780203315953, 'parameter_count': 89, 'epochs': 19, 'cap_limited_fits': 0}, 'jordan': {'hidden': 96, 'learning_rate': 0.01, 'mean_validation_mae': 1.3826419135235781, 'parameter_count': 385, 'epochs': 12, 'cap_limited_fits': 0}, 'multi': {'hidden': 18, 'learning_rate': 0.003, 'mean_val

,dataset,architecture,hidden,learning_rate,mean_validation_mae,parameter_count,epochs,cap_limited_fits
0,co2,elman,18,0.003,0.256284,379,286,0
1,co2,jordan,96,0.010,0.574700,385,69,0
2,co2,multi,18,0.010,0.270481,397,64,0
3,sunspots,elman,18,0.010,18.992991,379,14,0
4,sunspots,jordan,96,0.010,18.921695,385,10,0
5,sunspots,multi,18,0.010,18.993320,397,11,0
6,power,elman,18,0.010,0.211556,379,17,0
7,power,jordan,96,0.010,0.219082,385,28,0
8,power,multi,18,0.010,0.212229,397,15,0
9,retail,elman,8,0.010,1.395178,89,19,0


Saved 540 validation records and 15 selections


In [14]:
publish_stage("Archive frozen validation selection", [
    "output/primary/protocol_snapshot.json", "output/primary/selection",
    "output/primary/all_selected_before_test.json",
    "notebooks/exports/all_validation_runs.csv",
    "notebooks/exports/selected_configurations.csv"])

Review stage before committing: Archive frozen validation selection ['output/primary/protocol_snapshot.json', 'output/primary/selection', 'output/primary/all_selected_before_test.json', 'notebooks/exports/all_validation_runs.csv', 'notebooks/exports/selected_configurations.csv']


In [15]:
print('Stage 5: Final chronological evaluation')

Stage 5: Final chronological evaluation


In [16]:
if not (ROOT / "output/primary/all_selected_before_test.json").exists():
    raise RuntimeError("Selection must be frozen before final evaluation")
run_module("src.run_experiment", "--phase", "evaluation")
final_rows = []
for dataset in protocol["datasets"]:
    for path in sorted((ROOT / "output/primary/evaluation" / dataset).glob("*_s*.json")):
        row = json.loads(path.read_text())
        final_rows.append({"dataset": dataset, "architecture": row["architecture"],
                           "seed": row["seed"], "hidden": row["hidden"],
                           "learning_rate": row["learning_rate"],
                           "parameters": row["parameter_count"],
                           "epochs": row["epochs"],
                           "targets": len(row["indices"]),
                           **row["dataset_metrics"]})
if len(final_rows) != 45:
    raise AssertionError("Incomplete final evaluation")
final_table = pd.DataFrame(final_rows)
final_table.to_csv(EXPORT / "all_final_seed_runs.csv", index=False)
summaries = json.loads((ROOT / "output/primary/summary.json").read_text())
baseline_rows = [{"dataset": row["dataset"], "rule": rule, **metrics}
                 for row in summaries for rule, metrics in row["baselines"].items()]
pd.DataFrame(baseline_rows).to_csv(EXPORT / "all_baselines.csv", index=False)
display(final_table.groupby(["dataset", "architecture"])["mae"].agg(["mean", "std"]))
print("Saved", len(final_rows), "seed-level final results")

$ /opt/anaconda3/bin/python -m src.run_experiment --phase evaluation
670945, 'mase': 1.3543315227042185}, 'seasonal_naive': {'mae': 3.9775280898876404, 'rmse': 5.736831525768922, 'mase': 0.7620940375731892}}, 'networks': {'elman': {'mean_mae': 3.4574873586718957, 'sd_mae': 0.029536357389171677, 'mean_rmse': 5.1378478563856085, 'mean_mase': 0.6624542785071719, 'parameters': 89, 'epochs': 19}, 'jordan': {'mean_mae': 3.4158825704220974, 'sd_mae': 0.12024458715119873, 'mean_rmse': 5.121110879374728, 'mean_mase': 0.6544828046814365, 'parameters': 385, 'epochs': 12}, 'multi': {'mean_mae': 3.4950402330369976, 'sd_mae': 0.03979182512061191, 'mean_rmse': 5.237028545435383, 'mean_mase': 0.6696494059834905, 'parameters': 397, 'epochs': 14}}}
TEST rain {'dataset': 'rain', 'test_start_date': '2007-01-01', 'test_target_count': 228, 'baselines': {'persistence': {'mae': 39.57368421052631, 'rmse': 49.08976203692686, 'mase': 1.056479585391919}, 'seasonal_naive': {'mae': 42.041228070175436, 'rmse': 51.70

mean       std
dataset  architecture                     
co2      elman          0.327968  0.003399
         jordan         0.657749  0.012717
         multi          0.324429  0.006213
power    elman          0.168900  0.002275
         jordan         0.196301  0.026549
         multi          0.172555  0.002503
rain     elman         30.008737  0.526913
         jordan        31.454409  0.180260
         multi         30.884972  0.468841
retail   elman          3.457487  0.029536
         jordan         3.415883  0.120245
         multi          3.495040  0.039792
sunspots elman         13.412830  0.100191
         jordan        14.365927  0.464763
         multi         13.511974  0.131526

Saved 45 seed-level final results


In [17]:
publish_stage("Archive final tests and diagnostic baselines", [
    "output/primary/evaluation", "output/primary/summary.json",
    "notebooks/exports/all_final_seed_runs.csv",
    "notebooks/exports/all_baselines.csv"])

Review stage before committing: Archive final tests and diagnostic baselines ['output/primary/evaluation', 'output/primary/summary.json', 'notebooks/exports/all_final_seed_runs.csv', 'notebooks/exports/all_baselines.csv']


In [18]:
print('Stage 6: Independent verification and diagnostics')

Stage 6: Independent verification and diagnostics


In [19]:
run_module("src.verify_results")
run_module("src.diagnose_training")
verification = json.loads((ROOT / "output/diagnostics/verification.json").read_text())
curves = json.loads((ROOT / "output/diagnostics/training_curves.json").read_text())
display(pd.DataFrame([{
    "verification": verification["status"],
    "selection_records": verification["selection_records"],
    "final_records": verification["final_network_records"],
    "selected_development_fits": curves["selected_development_fits"],
    "training_loss_lower_after_checkpoint":
        curves["training_loss_lower_after_selected_checkpoint"]}]))

$ /opt/anaconda3/bin/python -m src.verify_results
{
  "status": "passed",
  "datasets": 5,
  "selection_records": 540,
  "final_network_records": 45,
  "raw_hashes": 5,
  "checks": [
    "chronology",
    "selection minimum",
    "selected epoch",
    "scaler scope",
    "parameter count",
    "target alignment",
    "MAE",
    "RMSE",
    "MASE",
    "baselines"
  ]
}

$ /opt/anaconda3/bin/python -m src.diagnose_training
{'selected_development_fits': 135, 'training_loss_lower_after_selected_checkpoint': 131, 'all_tail_mae_increases_by_construction': True}



,verification,selection_records,final_records,selected_development_fits,training_loss_lower_after_checkpoint
0,passed,540,45,135,131


In [20]:
publish_stage("Archive verification diagnostics and report", [
    "output/diagnostics/verification.json", "output/diagnostics/training_curves.json",
    "report/main.tex", "report/generated", "report/27187314RW441assignment3.pdf"])

Review stage before committing: Archive verification diagnostics and report ['output/diagnostics/verification.json', 'output/diagnostics/training_curves.json', 'report/main.tex', 'report/generated', 'report/27187314RW441assignment3.pdf']


In [21]:
print('Remote upload: Push reviewed commits only')

Remote upload: Push reviewed commits only


In [22]:
print('Stage 7: Post-hoc supplementary diagnostics and revised report')

Stage 7: Post-hoc supplementary diagnostics and revised report


In [23]:
run_module("src.supplementary_analysis")
paired = json.loads((ROOT / "output/diagnostics/paired_errors.json").read_text())
capacity = json.loads((ROOT / "output/diagnostics/capacity_sensitivity.json").read_text())
fit = json.loads((ROOT / "output/diagnostics/fit_sensitivity.json").read_text())
stationarity = json.loads((ROOT / "output/diagnostics/stationarity_sensitivity.json").read_text())
regime = json.loads((ROOT / "output/diagnostics/regime_difficulty.json").read_text())
paired_table = pd.DataFrame(paired["results"])
capacity_table = pd.DataFrame([{
    "dataset": row["dataset"],
    "small_leader": row["budgets"][0]["leader"],
    "small_validation_mae": row["budgets"][0]["leader_mean_validation_mae"],
    "large_leader": row["budgets"][1]["leader"],
    "large_validation_mae": row["budgets"][1]["leader_mean_validation_mae"]}
    for row in capacity["results"]])
fit_table = pd.DataFrame([{key: value for key, value in row.items() if key != "rows"}
                          for row in fit["results"]])
stationarity_table = pd.DataFrame([{
    "dataset": row["dataset"],
    "level_adf_trend_p": row["series"]["level"]["tests"]["ct"]["adf_unit_root_null_p"],
    "level_kpss_trend_p": row["series"]["level"]["tests"]["ct"]["kpss_stationarity_null_p"],
    "modelled_adf_trend_p": row["series"]["modelled_target"]["tests"]["ct"]["adf_unit_root_null_p"],
    "modelled_kpss_trend_p": row["series"]["modelled_target"]["tests"]["ct"]["kpss_stationarity_null_p"]}
    for row in stationarity["results"]])
regime_table = pd.DataFrame(regime["results"])
paired_table.to_csv(EXPORT / "paired_error_diagnostic.csv", index=False)
capacity_table.to_csv(EXPORT / "capacity_diagnostic.csv", index=False)
fit_table.to_csv(EXPORT / "fit_diagnostic.csv", index=False)
stationarity_table.to_csv(EXPORT / "stationarity_sensitivity.csv", index=False)
regime_table.to_csv(EXPORT / "regime_difficulty.csv", index=False)
display(paired_table[["dataset", "winner", "runner", "relative_advantage_percent",
                      "paired_seed_wins"]])
display(capacity_table)
display(fit_table)
display(stationarity_table)
display(regime_table[["dataset", "test_to_validation_persistence_mae_ratio",
                      "test_to_validation_standard_deviation_ratio"]])
run_module("src.make_report_assets")

$ /opt/anaconda3/bin/python -m src.supplementary_analysis
Supplementary diagnostics generated from frozen records and development data



,dataset,winner,runner,relative_advantage_percent,paired_seed_wins
0,co2,multi,elman,1.079056,3
1,sunspots,elman,multi,0.733750,2
2,power,elman,multi,2.118214,2
3,retail,jordan,elman,1.203324,2
4,rain,elman,multi,2.837089,3


,dataset,small_leader,small_validation_mae,large_leader,large_validation_mae
0,co2,multi,0.283075,elman,0.256284
1,sunspots,multi,19.952586,jordan,18.921695
2,power,elman,0.238057,elman,0.211556
3,retail,elman,1.395178,jordan,1.382642
4,rain,jordan,29.695070,elman,29.427084


,dataset,fit_count,median_best_epoch,median_train_reduction_to_checkpoint_percent,median_tail_reduction_to_checkpoint_percent,median_train_reduction_after_checkpoint_percent,median_tail_increase_after_checkpoint_percent,train_loss_lower_after_checkpoint_count
0,co2,27,79.0,93.393220,75.376457,2.407062,0.959598,26
1,sunspots,27,13.0,79.340209,47.811830,6.252148,2.910278,27
2,power,27,21.0,45.125707,12.048038,9.652442,4.881386,24
3,retail,27,15.0,67.776157,18.028758,19.518634,6.490827,27
4,rain,27,10.0,11.027935,5.902230,5.513760,2.069998,27


,dataset,level_adf_trend_p,level_kpss_trend_p,modelled_adf_trend_p,modelled_kpss_trend_p
0,co2,8.995515e-01,0.010000,9.880021e-10,0.100000
1,sunspots,3.571357e-09,0.100000,3.571357e-09,0.100000
2,power,1.221066e-01,0.031525,1.221066e-01,0.031525
3,retail,2.522042e-01,0.010000,3.637256e-01,0.017285
4,rain,1.723970e-08,0.100000,1.723970e-08,0.100000


,dataset,test_to_validation_persistence_mae_ratio,test_to_validation_standard_deviation_ratio
0,co2,1.078217,2.205988
1,sunspots,0.620403,0.742761
2,power,0.681028,1.029163
3,retail,1.012092,1.014394
4,rain,1.015144,1.029328


$ /opt/anaconda3/bin/python -m src.make_report_assets
Report tables and figures generated



CompletedProcess(args=['/opt/anaconda3/bin/python', '-m', 'src.make_report_assets'], returncode=0, stdout='Report tables and figures generated\n', stderr='')

In [24]:
import shutil

for command in (["pdflatex", "-interaction=nonstopmode", "-halt-on-error", "main.tex"],
                ["bibtex", "main"],
                ["pdflatex", "-interaction=nonstopmode", "-halt-on-error", "main.tex"],
                ["pdflatex", "-interaction=nonstopmode", "-halt-on-error", "main.tex"]):
    result = subprocess.run(command, cwd=ROOT / "report", text=True, capture_output=True)
    if result.returncode:
        print(result.stdout[-3000:])
        print(result.stderr[-1000:])
        raise RuntimeError("Report build failed")
shutil.copy2(ROOT / "report/main.pdf",
             ROOT / "report/27187314RW441assignment3.pdf")
print("report/27187314RW441assignment3.pdf")

report/27187314RW441assignment3.pdf


In [25]:
publish_stage("Add post-hoc diagnostic code", [
    "src/supplementary_analysis.py", "src/make_report_assets.py"])

Review stage before committing: Add post-hoc diagnostic code ['src/supplementary_analysis.py', 'src/make_report_assets.py']
